# Assignment 02 - Colorectal Histopathology CNN Classifier

CAME | PyTorch + Google Colab T4 GPU | custom CNN trained **from scratch** on `1aurent/NCT-CRC-HE` (`NCT_CRC_HE_100K`).

**How this notebook follows the assignment PDF**

1. Sections 2-8 write `assignment02.py` one part at a time with `%%writefile`: the 9 required functions, `TRAINED_MODEL_B64`, `encode_model_to_b64`, `main()` and the CMS ending exactly as written in the PDF. The submitted `.py` is therefore exactly the code in this notebook.
2. Section 9 imports the module and checks the 0-mark gate rules (import does not train, >= 3 `Conv2d`, allowed layers only, logits `(N, 9)`, 25K-500K parameters).
3. Sections 10-11 train on the GPU with `main()`: best validation checkpoint -> `best_model.pt` -> `encode_model_to_b64()` -> `load_trained_model(cpu)` check -> `assignment02_submission.py` with `TRAINED_MODEL_B64` filled in.
4. Section 12 plots the results. Section 13 simulates CMS Stage A (`CAME_DATA_DIR` ImageFolder, 2 epochs on CPU). Section 14 checks the submission file the way Stage B uses it, then downloads it.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**, then run the cells in order. After editing a code cell in sections 2-8, re-run sections 2-8 in order (section 2 recreates the file from scratch).

**Submit:** rename `assignment02_submission.py` to `assignment02.py`, upload that `.py` file to CMS (never this `.ipynb`) and press Play once.

## 0. Install libraries and check the GPU

This is the first cell given in the PDF.

In [ ]:
!pip install -q torch torchvision datasets pillow
import torch
print('CUDA available:', torch.cuda.is_available(), '| Device:',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Working directory

With `USE_DRIVE = True` the code, the cached patch subset (`nct_crc_he_100k_subset_2000.pt`) and the submission file are kept on Google Drive, so a Colab reset does not lose them. If that subset is already cached in the folder from an earlier run, loading the data takes seconds instead of streaming it again.

In [ ]:
import os, sys

USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = '/content/drive/MyDrive/came_a02'
else:
    WORK_DIR = '/content/came_a02'

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print('Working directory:', os.getcwd())

## 2. Module header, imports and constants

`TRAINED_MODEL_B64` starts empty here. `main()` fills it in automatically inside `assignment02_submission.py` after training (PDF Task 5, step 4).

In [ ]:
%%writefile assignment02.py
"""
CAME Assignment 02 - Colorectal Histopathology CNN Classifier (PyTorch).

Custom CNN trained FROM SCRATCH (random initialisation, no pretrained weights) on a
Google Colab T4 GPU to classify 64x64 H&E colorectal tissue patches into 9 classes:
    0 ADI, 1 BACK, 2 DEB, 3 LYM, 4 MUC, 5 MUS, 6 NORM, 7 STR, 8 TUM

Colab training setup (used for the weights submitted in TRAINED_MODEL_B64):
    dataset   : 1aurent/NCT-CRC-HE, split NCT_CRC_HE_100K
    subset    : deterministic balanced subset, 2000 patches per class (18,000)
    split     : 85 % train (15,300) / 15 % validation (2,700), seed 42
    training  : AdamW(lr=1e-3, weight_decay=1e-4), CrossEntropyLoss, torch.manual_seed(42);
                the notebook sets CAME_EPOCHS=25 and CAME_BATCH_SIZE=32
                (PDF defaults when unset: 12 epochs, batch size 32)
    model     : 6 x Conv2d funnel CNN (3 -> 32 -> 64 -> 128 -> 256), Dropout 0.5,
                437,481 trainable parameters

Colab workflow
    1. Runtime -> Change runtime type -> T4 GPU.
    2. Run main() (the notebook, or `!python assignment02.py`). It trains, restores the
       best validation checkpoint, runs encode_model_to_b64(), checks
       load_trained_model(cpu) and writes `assignment02_submission.py`: this file with
       TRAINED_MODEL_B64 filled in.
    3. Rename `assignment02_submission.py` to `assignment02.py` and upload it to CMS.
       Never upload a copy whose TRAINED_MODEL_B64 is empty.

Importing this module does NOT train and does NOT download data.
"""

import base64
import copy
import io
import os
import re
import time
import zlib
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.datasets import ImageFolder

# ---------------------------------------------------------------------------
# Base64 string of the Colab-trained best_model.pt (fp16 state_dict -> zlib -> base64).
# Empty in the source file; main() writes assignment02_submission.py with it filled in.
# ---------------------------------------------------------------------------
TRAINED_MODEL_B64 = """
"""

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
NUM_CLASSES = 9
IMAGE_SIZE = 64
NORM_MEAN = (0.5, 0.5, 0.5)
NORM_STD = (0.5, 0.5, 0.5)
SEED = 42
VAL_FRACTION = 0.15
PATCHES_PER_CLASS = 2000  # Colab subset size per class (override with CAME_PER_CLASS)

HF_DATASET_ID = "1aurent/NCT-CRC-HE"
HF_SPLIT = "NCT_CRC_HE_100K"

## 3. Task 1 - Data preprocessing and loaders (15 marks)

- `official_classes()` returns the exact 9 classes in PDF order.
- `make_transforms(True)`: Resize(64, 64) -> RandomHorizontalFlip -> RandomVerticalFlip -> ToTensor -> Normalize(0.5). `make_transforms(False)`: no random flips.
- `build_datasets()`: if `CAME_DATA_DIR` is set (CMS) it uses only `ImageFolder(train)` / `ImageFolder(validation)`, with no `datasets` import and no internet. Otherwise (Colab) it streams `1aurent/NCT-CRC-HE`, split `NCT_CRC_HE_100K`, one parquet shard at a time, builds a deterministic balanced subset (`CAME_PER_CLASS`, default 2000 per class), caches it, and splits it 85 % / 15 % with seed 42.
- `make_loaders()`: `shuffle=True` for train, `shuffle=False` for validation, `num_workers=int(os.environ.get('CAME_WORKERS', 0))`.

In [ ]:
%%writefile -a assignment02.py


# ===========================================================================
# Task 1 - Data preprocessing & loaders
# ===========================================================================
def official_classes() -> list[str]:
    return ["ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM"]


def make_transforms(train: bool):
    if train:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize(NORM_MEAN, NORM_STD),
        ])
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD),
    ])


class NCTPatchDataset(Dataset):
    """In-memory dataset of 64x64 RGB patches (Colab / Hugging Face path).

    images: uint8 tensor (N, 64, 64, 3); labels: long tensor (N,).
    Each item is (tensor_3x64x64, int_label_0_to_8).
    """

    def __init__(self, images: torch.Tensor, labels: torch.Tensor, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self) -> int:
        return int(self.labels.numel())

    def __getitem__(self, idx):
        from PIL import Image

        img = Image.fromarray(self.images[idx].numpy())  # (64, 64, 3) uint8 -> RGB
        return self.transform(img), int(self.labels[idx])


def _label_mapper(features):
    """Map Hugging Face labels (ClassLabel int or str) to official indices 0..8."""
    classes = official_classes()
    remap = None
    try:
        names = list(getattr(features["label"], "names", None) or [])
        if names:
            remap = {i: classes.index(n.strip().upper()) for i, n in enumerate(names)
                     if n.strip().upper() in classes}
    except Exception:
        remap = None

    def to_idx(value) -> int:
        if isinstance(value, str):
            return classes.index(value.strip().upper())
        value = int(value)
        if remap is not None:
            return remap.get(value, -1)
        return value

    return to_idx


def _patch_to_uint8(image_field) -> torch.Tensor:
    """Decode one HF image field (PIL or {'bytes', 'path'}) to a (64, 64, 3) uint8 tensor."""
    import numpy as np
    from PIL import Image

    if isinstance(image_field, dict):
        if image_field.get("bytes") is not None:
            img = Image.open(io.BytesIO(image_field["bytes"]))
        else:
            img = Image.open(image_field["path"])
    else:
        img = image_field
    img = img.convert("RGB")
    img = transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))(img)
    return torch.from_numpy(np.array(img, dtype=np.uint8).copy())


def _list_hf_shards() -> list[str]:
    """Parquet shard paths of the NCT_CRC_HE_100K split (relative to the dataset repo)."""
    try:
        from huggingface_hub import HfFileSystem

        fs = HfFileSystem()
        files = sorted(fs.glob(f"datasets/{HF_DATASET_ID}/data/{HF_SPLIT}-*.parquet"))
        shards = ["data/" + f.rsplit("/", 1)[-1] for f in files]
        if shards:
            return shards
    except Exception as exc:  # network dependent
        print(f"[data] could not list shards via HfFileSystem ({exc}); using default pattern")
    return [f"data/{HF_SPLIT}-{i:05d}-of-00031-*.parquet" for i in range(31)]


def _collect_streaming(per_class: int):
    """Stream NCT_CRC_HE_100K one parquet shard at a time until every class has
    `per_class` patches (no full ~15 GB download, bounded RAM, fixed read order)."""
    import gc

    from datasets import Image as HFImage
    from datasets import load_dataset

    shards = _list_hf_shards()
    default_rows = max(64, (per_class * NUM_CLASSES * 13) // (10 * len(shards)))
    rows_per_round = int(os.environ.get("CAME_ROWS_PER_SHARD", default_rows))
    max_rounds = int(os.environ.get("CAME_MAX_ROUNDS", 12))
    print(f"[data] streaming {HF_DATASET_ID}:{HF_SPLIT}: {len(shards)} shards, "
          f"{rows_per_round} rows per shard per round")

    buckets = {c: [] for c in range(NUM_CLASSES)}
    exhausted = set()
    to_idx = None
    scanned = 0
    t0 = time.time()

    def all_full() -> bool:
        return all(len(buckets[c]) >= per_class for c in range(NUM_CLASSES))

    for rnd in range(max_rounds):
        if all_full() or len(exhausted) == len(shards):
            break
        for si, shard in enumerate(shards):
            if all_full():
                break
            if si in exhausted:
                continue
            ds = load_dataset(HF_DATASET_ID, data_files={HF_SPLIT: shard},
                              split=HF_SPLIT, streaming=True)
            try:
                ds = ds.cast_column("image", HFImage(decode=False))
            except Exception:
                pass
            if to_idx is None:
                to_idx = _label_mapper(ds.features)
            if rnd:
                ds = ds.skip(rnd * rows_per_round)
            n_read = 0
            for example in ds.take(rows_per_round):
                n_read += 1
                y = to_idx(example["label"])
                if 0 <= y < NUM_CLASSES and len(buckets[y]) < per_class:
                    buckets[y].append(_patch_to_uint8(example["image"]))
            scanned += n_read
            if n_read < rows_per_round:
                exhausted.add(si)
            del ds
            gc.collect()
            counts = [len(buckets[c]) for c in range(NUM_CLASSES)]
            print(f"[data] round {rnd + 1} shard {si + 1}/{len(shards)} | scanned={scanned} "
                  f"per-class={counts} ({time.time() - t0:.0f}s)")
    return buckets


def _collect_full(per_class: int):
    """Download the full NCT_CRC_HE_100K split and draw a seeded balanced subset."""
    from datasets import Image as HFImage
    from datasets import load_dataset

    ds = load_dataset(HF_DATASET_ID, split=HF_SPLIT)
    to_idx = _label_mapper(ds.features)
    labels = torch.tensor([to_idx(v) for v in ds["label"]], dtype=torch.long)
    g = torch.Generator().manual_seed(SEED)
    chosen = []
    for c in range(NUM_CLASSES):
        idx = (labels == c).nonzero(as_tuple=False).view(-1)
        idx = idx[torch.randperm(idx.numel(), generator=g)][:per_class]
        chosen.extend(idx.tolist())
    chosen.sort()
    sub = ds.select(chosen)
    try:
        sub = sub.cast_column("image", HFImage(decode=False))
    except Exception:
        pass
    buckets = {c: [] for c in range(NUM_CLASSES)}
    for example in sub:
        y = to_idx(example["label"])
        buckets[y].append(_patch_to_uint8(example["image"]))
    return buckets


def _load_hf_subset(per_class: int):
    """Return (images uint8 (N,64,64,3), labels long (N,)): balanced subset, cached on disk."""
    mode = os.environ.get("CAME_HF_MODE", "stream").lower()
    cache = Path(os.environ.get("CAME_CACHE", f"nct_crc_he_100k_subset_{per_class}.pt"))

    if cache.exists():
        print(f"[data] loading cached subset from {cache}")
        blob = torch.load(cache, map_location="cpu", weights_only=True)
        return blob["images"], blob["labels"]

    buckets = _collect_full(per_class) if mode == "full" else _collect_streaming(per_class)

    images, labels = [], []
    for c in range(NUM_CLASSES):
        if not buckets[c]:
            raise RuntimeError(f"No patches collected for class {official_classes()[c]}")
        images.extend(buckets[c])
        labels.extend([c] * len(buckets[c]))
    images = torch.stack(images).to(torch.uint8)
    labels = torch.tensor(labels, dtype=torch.long)
    try:
        torch.save({"images": images, "labels": labels}, cache)
        print(f"[data] cached subset to {cache}")
    except Exception as exc:
        print(f"[data] could not cache subset ({exc})")
    return images, labels


def _select_and_split(labels: torch.Tensor, per_class: int,
                      val_fraction: float = VAL_FRACTION, seed: int = SEED):
    """Deterministic balanced selection + per-class 85 / 15 train / validation split."""
    g = torch.Generator().manual_seed(seed)
    train_idx, val_idx = [], []
    for c in range(NUM_CLASSES):
        idx = (labels == c).nonzero(as_tuple=False).view(-1)
        idx = idx[torch.randperm(idx.numel(), generator=g)][:per_class]
        n_val = int(round(idx.numel() * val_fraction))
        val_idx.append(idx[:n_val])
        train_idx.append(idx[n_val:])
    return torch.cat(train_idx), torch.cat(val_idx)


def build_datasets():
    data_dir = os.environ.get("CAME_DATA_DIR")
    if data_dir:
        # CMS: local ImageFolder only (no `datasets` import, no internet).
        train_dataset = ImageFolder(os.path.join(data_dir, "train"), transform=make_transforms(True))
        val_dataset = ImageFolder(os.path.join(data_dir, "validation"), transform=make_transforms(False))
        return train_dataset, val_dataset

    # Google Colab: 1aurent/NCT-CRC-HE, split NCT_CRC_HE_100K.
    per_class = int(os.environ.get("CAME_PER_CLASS", PATCHES_PER_CLASS))
    images, labels = _load_hf_subset(per_class)
    train_idx, val_idx = _select_and_split(labels, per_class)
    train_dataset = NCTPatchDataset(images[train_idx], labels[train_idx], make_transforms(True))
    val_dataset = NCTPatchDataset(images[val_idx], labels[val_idx], make_transforms(False))
    print(f"[data] train={len(train_dataset)} val={len(val_dataset)}")
    return train_dataset, val_dataset


def make_loaders(train_dataset, val_dataset, batch_size: int):
    num_workers = int(os.environ.get("CAME_WORKERS", 0))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers)
    return train_loader, val_loader

## 4. Task 2 - Custom CNN from scratch (15 marks)

A plain `nn.Sequential` built only from allowed layers, with random initialisation and no pretrained weights. Channels funnel 3 -> 32 -> 64 -> 128 -> 256 while the spatial size goes 64 -> 32 -> 16 -> 8 -> 4: 6 x `Conv2d(3x3, padding=1)` + BatchNorm + ReLU, with 2x2 MaxPool. The head is GlobalAvgPool -> Flatten -> Dropout(0.5) -> `Linear(256, 9)` and returns raw logits (no softmax). **437,481** trainable parameters.

In [ ]:
%%writefile -a assignment02.py


# ===========================================================================
# Task 2 - Custom CNN (from scratch)
# ===========================================================================
def _conv_bn_relu(in_ch: int, out_ch: int) -> list:
    return [
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    ]


def build_model(num_classes: int = 9) -> torch.nn.Module:
    """Funnel CNN 3 -> 32 -> 64 -> 128 -> 256 channels, 64 -> 4 spatial.

    Allowed layers only (Sequential, Conv2d, BatchNorm2d, ReLU, MaxPool2d,
    AdaptiveAvgPool2d, Flatten, Dropout, Linear). Returns raw logits (N, num_classes).
    Trainable parameters (num_classes=9): 437,481.
    """
    features = nn.Sequential(
        # Block 1: 3 -> 32, 64x64 -> 32x32
        nn.Sequential(*_conv_bn_relu(3, 32), *_conv_bn_relu(32, 32), nn.MaxPool2d(2, 2)),
        # Block 2: 32 -> 64, 32x32 -> 16x16
        nn.Sequential(*_conv_bn_relu(32, 64), *_conv_bn_relu(64, 64), nn.MaxPool2d(2, 2)),
        # Block 3: 64 -> 128, 16x16 -> 8x8
        nn.Sequential(*_conv_bn_relu(64, 128), nn.MaxPool2d(2, 2)),
        # Block 4: 128 -> 256, 8x8 -> 4x4
        nn.Sequential(*_conv_bn_relu(128, 256), nn.MaxPool2d(2, 2)),
    )
    classifier = nn.Sequential(
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Dropout(p=0.5),
        nn.Linear(256, num_classes),
    )
    return nn.Sequential(features, classifier)


def count_parameters(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 5. Task 3 - Training loop, validation and best checkpoint restoration (25 marks)

Exactly as the PDF specifies: `torch.manual_seed(42)`, `build_model(9).to(device)`, `CrossEntropyLoss`, `AdamW(lr=1e-3, weight_decay=1e-4)`, sample-weighted loss and accuracy, a deep copy of the best `state_dict` (higher `val_acc`, or equal within 1e-12 with lower `val_loss`), `load_state_dict(best_state)` at the end, and `(model, history)` returned.

In [ ]:
%%writefile -a assignment02.py


# ===========================================================================
# Task 3 - Training loop, validation & best checkpoint restoration
# ===========================================================================
def train_model(train_loader, val_loader, epochs: int, device: torch.device):
    torch.manual_seed(42)
    model = build_model(9).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = -1.0
    best_val_loss = float("inf")
    history = []

    for epoch in range(epochs):
        t0 = time.time()

        # ---- training phase ----
        model.train()
        loss_sum, correct, seen = 0.0, 0, 0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device).long()
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            bs = labels.size(0)
            loss_sum += loss.item() * bs
            correct += (logits.argmax(dim=1) == labels).sum().item()
            seen += bs
        train_loss = loss_sum / max(seen, 1)
        train_acc = correct / max(seen, 1)

        # ---- validation phase ----
        model.eval()
        loss_sum, correct, seen = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device).long()
                logits = model(images)
                loss = criterion(logits, labels)
                bs = labels.size(0)
                loss_sum += loss.item() * bs
                correct += (logits.argmax(dim=1) == labels).sum().item()
                seen += bs
        val_loss = loss_sum / max(seen, 1)
        val_acc = correct / max(seen, 1)

        # ---- checkpoint tracking ----
        if abs(val_acc - best_val_acc) <= 1e-12:
            improved = val_loss < best_val_loss
        else:
            improved = val_acc > best_val_acc
        if improved:
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        history.append({
            "train_loss": float(train_loss),
            "train_acc": float(train_acc),
            "val_loss": float(val_loss),
            "val_acc": float(val_acc),
        })
        print(f"Epoch {epoch + 1:02d}/{epochs} | train_loss={train_loss:.4f} "
              f"train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
              f"{' *best*' if improved else ''} | {time.time() - t0:.1f}s")

    model.load_state_dict(best_state)
    return model, history

## 6. Task 4 - Inference and evaluation metrics (10 marks)

`predict()` returns a 1-D `torch.long` CPU tensor in loader order. `classification_report()` returns `accuracy`, `macro_recall` (averaged only over classes that appear in `y_true`) and a `(9, 9)` `torch.long` confusion matrix with rows = true and columns = predicted.

In [ ]:
%%writefile -a assignment02.py


# ===========================================================================
# Task 4 - Inference & metrics
# ===========================================================================
def predict(model: torch.nn.Module, loader, device: torch.device) -> torch.Tensor:
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            images = batch[0] if isinstance(batch, (list, tuple)) else batch
            images = images.to(device)
            logits = model(images)
            preds.append(logits.argmax(dim=1).detach().cpu().long())
    if not preds:
        return torch.empty(0, dtype=torch.long)
    return torch.cat(preds).long()


def classification_report(y_true: torch.Tensor, y_pred: torch.Tensor) -> dict:
    n = NUM_CLASSES
    y_true = torch.as_tensor(y_true).detach().cpu().long().view(-1)
    y_pred = torch.as_tensor(y_pred).detach().cpu().long().view(-1)
    if y_true.numel() != y_pred.numel():
        raise ValueError(f"y_true has {y_true.numel()} items but y_pred has {y_pred.numel()}")

    confusion = torch.zeros((n, n), dtype=torch.long)
    if y_true.numel() > 0:
        valid = (y_true >= 0) & (y_true < n) & (y_pred >= 0) & (y_pred < n)
        flat = y_true[valid] * n + y_pred[valid]
        confusion = torch.bincount(flat, minlength=n * n).reshape(n, n).long()
        accuracy = (y_true == y_pred).double().mean().item()
    else:
        accuracy = 0.0

    support = confusion.sum(dim=1)
    present = support > 0
    if present.any():
        recalls = confusion.diag()[present].double() / support[present].double()
        macro_recall = recalls.mean().item()
    else:
        macro_recall = 0.0

    return {
        "accuracy": float(accuracy),
        "macro_recall": float(macro_recall),
        "confusion": confusion,
    }

## 7. Task 5 - Embedding the Colab-trained weights (35 marks)

`encode_model_to_b64()` and `load_trained_model()` are the PDF helper code. PDF step 4 says to paste the base64 string into `TRAINED_MODEL_B64`; here `main()` does that automatically: `_write_submission_file()` writes `assignment02_submission.py`, a copy of this file with `TRAINED_MODEL_B64` filled in.

In [ ]:
%%writefile -a assignment02.py


# ===========================================================================
# Task 5 - Embedding the Colab-trained model
# ===========================================================================
def encode_model_to_b64(model: torch.nn.Module) -> str:
    """Run this on Colab after training; main() calls it and fills TRAINED_MODEL_B64 for you."""
    state = {k: v.detach().cpu().half() for k, v in model.state_dict().items()}
    buf = io.BytesIO()
    torch.save(state, buf)
    compressed = zlib.compress(buf.getvalue(), level=9)
    return base64.b64encode(compressed).decode("ascii")


def load_trained_model(device: torch.device) -> torch.nn.Module:
    """Rebuilds build_model(9) and loads TRAINED_MODEL_B64 on CMS."""
    model = build_model(9)
    raw = zlib.decompress(base64.b64decode(TRAINED_MODEL_B64.strip()))
    state = torch.load(io.BytesIO(raw), map_location="cpu", weights_only=True)
    state = {k: v.float() if v.is_floating_point() else v for k, v in state.items()}
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    return model


def _write_submission_file(b64: str, out_path: str = "assignment02_submission.py"):
    """Copy this file with TRAINED_MODEL_B64 filled in (wrapped at 100 characters;
    base64.b64decode ignores the newlines). Returns the written path or None."""
    try:
        source = Path(__file__).read_text(encoding="utf-8")
    except Exception as exc:
        print(f"[export] could not read source file ({exc}); paste trained_model_b64.txt manually")
        return None
    wrapped = "\n".join(b64[i:i + 100] for i in range(0, len(b64), 100))
    new_source, n_subs = re.subn(
        r'^TRAINED_MODEL_B64 = """.*?"""',
        lambda _m: 'TRAINED_MODEL_B64 = """\n' + wrapped + '\n"""',
        source, count=1, flags=re.DOTALL | re.MULTILINE,
    )
    if n_subs != 1:
        print("[export] TRAINED_MODEL_B64 placeholder not found; paste trained_model_b64.txt manually")
        return None
    Path(out_path).write_text(new_source, encoding="utf-8")
    print(f"[export] wrote {out_path}")
    return out_path

## 8. Colab entry point and the required CMS ending

`main()` runs only when it is called (`!python assignment02.py` or section 11 of this notebook), so importing the module never trains or downloads anything, and on CMS the ending runs the grader instead of `main()`. `_run_on_cms_if_needed()` and the `__main__` block are copied exactly from the PDF.

In [ ]:
%%writefile -a assignment02.py


# ===========================================================================
# Colab entry point: main() runs only when called (`python assignment02.py` or the
# notebook), never on import and never on CMS.
# ===========================================================================
def main():
    global TRAINED_MODEL_B64

    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("CUDA available:", torch.cuda.is_available(), "| Device:",
          torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

    # PDF defaults (batch size 32, 12 epochs). The notebook overrides them with
    # CAME_EPOCHS=25 and CAME_BATCH_SIZE=32, the settings used for the submitted weights.
    batch_size = int(os.environ.get("CAME_BATCH_SIZE", 32))
    epochs = int(os.environ.get("CAME_EPOCHS", 12))

    train_dataset, val_dataset = build_datasets()
    train_loader, val_loader = make_loaders(train_dataset, val_dataset, batch_size)
    print(f"Model parameters: {count_parameters(build_model(9)):,}")
    print(f"Training for {epochs} epochs, batch size {batch_size}")

    model, history = train_model(train_loader, val_loader, epochs, device)

    y_true = torch.cat([labels.long() for _, labels in val_loader])
    report = classification_report(y_true, predict(model, val_loader, device))
    print(f"Best checkpoint | val accuracy={report['accuracy']:.4f} "
          f"macro_recall={report['macro_recall']:.4f}")
    print("Confusion matrix (rows=true, cols=pred):", official_classes())
    print(report["confusion"])

    torch.save(model.state_dict(), "best_model.pt")
    b64 = encode_model_to_b64(model)
    Path("trained_model_b64.txt").write_text(b64, encoding="ascii")
    print("Saved best_model.pt and trained_model_b64.txt")

    # Verify the CMS loading path on CPU (fp16 round trip).
    TRAINED_MODEL_B64 = b64
    cpu = torch.device("cpu")
    cpu_report = classification_report(y_true, predict(load_trained_model(cpu), val_loader, cpu))
    print(f"load_trained_model(cpu) OK | val accuracy={cpu_report['accuracy']:.4f} "
          f"macro_recall={cpu_report['macro_recall']:.4f}")

    submission = _write_submission_file(b64)

    # Returned for notebook checks / plots; ignored when run as a script.
    return {
        "model": model,
        "history": history,
        "report": report,
        "cpu_report": cpu_report,
        "val_loader": val_loader,
        "submission": submission,
    }


def _run_on_cms_if_needed() -> bool:
    zpath = Path("cms_eval_data.zip")
    if not zpath.exists():
        return False
    import sys, runpy, zipfile
    sys.modules["a02_cnn"] = sys.modules["__main__"]
    with zipfile.ZipFile(zpath) as archive:
        archive.extract("cms_grader.py", ".")
    runpy.run_path("cms_grader.py", run_name="__main__")
    return True


if __name__ == "__main__":
    if not _run_on_cms_if_needed():
        main()

## 9. Import the module and run the gate checks

These are the automatic 0-mark gate rules from the PDF, checked locally. Every `assert` must pass.

In [ ]:
import importlib, time
import torch
import torch.nn as nn

t0 = time.time()
import assignment02 as a02
a02 = importlib.reload(a02)  # pick up the file written above
print(f'Import took {time.time() - t0:.2f}s (importing must not train or download)')

# Task 1: classes and transforms
assert a02.official_classes() == ["ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM"]
train_names = [type(t).__name__ for t in a02.make_transforms(True).transforms]
eval_names = [type(t).__name__ for t in a02.make_transforms(False).transforms]
print('train transforms:', train_names)
print('eval transforms :', eval_names)
assert 'RandomHorizontalFlip' in train_names
assert not any(name.startswith('Random') for name in eval_names)

# Task 2: model gate
model = a02.build_model(9)
allowed = (nn.Sequential, nn.Conv2d, nn.BatchNorm2d, nn.ReLU, nn.MaxPool2d,
           nn.AdaptiveAvgPool2d, nn.Flatten, nn.Dropout, nn.Linear)
bad = sorted({type(m).__name__ for m in model.modules() if type(m) not in allowed})
n_conv = sum(isinstance(m, nn.Conv2d) for m in model.modules())
n_params = a02.count_parameters(model)
with torch.no_grad():
    out = model.eval()(torch.randn(4, 3, 64, 64))
print(f'Conv2d layers: {n_conv} | parameters: {n_params:,} | output: {tuple(out.shape)} | disallowed layers: {bad}')
assert isinstance(model, nn.Module) and n_conv >= 3 and not bad
assert tuple(out.shape) == (4, 9)
assert 25_000 <= n_params <= 500_000

# Task 4: metrics on a tiny known example
rep = a02.classification_report(torch.tensor([0, 0, 1, 2]), torch.tensor([0, 1, 1, 2]))
assert abs(rep['accuracy'] - 0.75) < 1e-9
assert abs(rep['macro_recall'] - (0.5 + 1.0 + 1.0) / 3) < 1e-9
assert rep['confusion'].shape == (9, 9) and rep['confusion'].dtype == torch.long
print('All gate checks passed.')

## 10. Training settings

These settings gave the best validation macro recall in earlier Colab runs: 25 epochs, batch size 32, and 2000 patches per class (15,300 train / 2,700 validation). They override the PDF defaults in `main()` (12 epochs, batch size 32). The optimizer, learning rate, weight decay, loss and seed are fixed inside `train_model()` exactly as the PDF requires, so they are not set here.

`CAME_DATA_DIR` must **not** be set on Colab, because it is the switch that tells `build_datasets()` it is running on CMS.

In [ ]:
import os

os.environ['CAME_EPOCHS'] = '25'
os.environ['CAME_BATCH_SIZE'] = '32'
os.environ['CAME_PER_CLASS'] = '2000'
os.environ['CAME_HF_MODE'] = 'stream'   # 'full' downloads the whole ~15 GB split
os.environ['CAME_WORKERS'] = '2'
os.environ.pop('CAME_DATA_DIR', None)

for k in ['CAME_EPOCHS', 'CAME_BATCH_SIZE', 'CAME_PER_CLASS', 'CAME_HF_MODE', 'CAME_WORKERS']:
    print(f'{k} = {os.environ[k]}')

## 11. Train on the GPU

`main()` loads the data, trains, restores the best checkpoint, evaluates it, saves `best_model.pt` and `trained_model_b64.txt`, checks `load_trained_model(cpu)` and writes `assignment02_submission.py`. The same thing can be run as a script with `!python assignment02.py`.

In [ ]:
a02 = importlib.reload(a02)
result = a02.main()

## 12. Learning curves and confusion matrix

In [ ]:
import matplotlib.pyplot as plt

h = result['history']
epochs_axis = range(1, len(h) + 1)

# Same best-epoch rule as train_model()
best_i, best_acc, best_loss = 0, -1.0, float('inf')
for i, r in enumerate(h):
    better = (r['val_loss'] < best_loss) if abs(r['val_acc'] - best_acc) <= 1e-12 else (r['val_acc'] > best_acc)
    if better:
        best_i, best_acc, best_loss = i, r['val_acc'], r['val_loss']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key, title in [(axes[0], 'loss', 'Cross-entropy loss'), (axes[1], 'acc', 'Accuracy')]:
    ax.plot(epochs_axis, [r[f'train_{key}'] for r in h], marker='o', label='train')
    ax.plot(epochs_axis, [r[f'val_{key}'] for r in h], marker='o', label='validation')
    ax.axvline(best_i + 1, color='gray', linestyle='--', label=f'best epoch ({best_i + 1})')
    ax.set_title(title)
    ax.set_xlabel('epoch')
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()
print(f"Best epoch {best_i + 1}: val_acc={best_acc:.4f}, val_loss={best_loss:.4f}")

In [ ]:
import numpy as np

classes = a02.official_classes()
conf = result['report']['confusion'].numpy()
norm = conf / np.clip(conf.sum(axis=1, keepdims=True), 1, None)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(norm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(classes)), classes, rotation=45)
ax.set_yticks(range(len(classes)), classes)
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, int(conf[i, j]), ha='center', va='center', fontsize=8,
                color='white' if norm[i, j] > 0.5 else 'black')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f"Validation confusion matrix | acc={result['report']['accuracy']:.3f} "
             f"macro recall={result['report']['macro_recall']:.3f}")
fig.colorbar(im, ax=ax, label='row-normalised (recall)')
plt.tight_layout()
plt.show()

for name, r in zip(classes, norm.diagonal()):
    print(f'{name:5s} recall = {r:.3f}')
print(f"CPU fp16 reload: acc={result['cpu_report']['accuracy']:.4f} "
      f"macro recall={result['cpu_report']['macro_recall']:.4f}")

## 13. Simulate CMS Stage A (65 marks)

On CMS, `CAME_DATA_DIR` points to a small `ImageFolder` (`train/` and `validation/`, one sub-folder per class), and the grader runs `build_datasets()`, `make_loaders()`, `build_model()`, `train_model()` for 2 epochs, `predict()` and `classification_report()` on CPU in about 25 seconds.

This cell builds a similar folder from a few validation patches and runs the same calls on CPU. It only tests the pipeline, so the accuracy here does not matter. `CAME_DATA_DIR` is removed again at the end.

In [ ]:
import shutil
from PIL import Image

SIM_DIR = '/content/cms_stage_a_sim'
shutil.rmtree(SIM_DIR, ignore_errors=True)
val_ds = result['val_loader'].dataset
classes = a02.official_classes()
for split, (start, n) in {'train': (0, 20), 'validation': (20, 10)}.items():
    for c, name in enumerate(classes):
        os.makedirs(f'{SIM_DIR}/{split}/{name}', exist_ok=True)
        for i in (val_ds.labels == c).nonzero().view(-1)[start:start + n].tolist():
            Image.fromarray(val_ds.images[i].numpy()).save(f'{SIM_DIR}/{split}/{name}/{i}.png')

os.environ['CAME_DATA_DIR'] = SIM_DIR
try:
    cpu = torch.device('cpu')
    t0 = time.time()
    tr_ds, va_ds = a02.build_datasets()
    assert tr_ds.classes == classes, tr_ds.classes  # ImageFolder order == official order
    tr_dl, va_dl = a02.make_loaders(tr_ds, va_ds, 32)
    sim_model, sim_hist = a02.train_model(tr_dl, va_dl, 2, cpu)
    y_true = torch.cat([y.long() for _, y in va_dl])
    y_pred = a02.predict(sim_model, va_dl, cpu)
    sim_rep = a02.classification_report(y_true, y_pred)
    elapsed = time.time() - t0
finally:
    os.environ.pop('CAME_DATA_DIR', None)

print(f'train={len(tr_ds)} val={len(va_ds)} | {elapsed:.1f}s on Colab CPU (CMS budget is about 25 s)')
assert len(sim_hist) == 2
assert set(sim_hist[0]) == {'train_loss', 'train_acc', 'val_loss', 'val_acc'}
assert y_pred.dtype == torch.long and y_pred.dim() == 1 and len(y_pred) == len(va_ds)
assert sim_rep['confusion'].shape == (9, 9) and sim_rep['confusion'].dtype == torch.long
print('Stage A simulation passed:', {k: round(v, 3) for k, v in sim_rep.items() if k != 'confusion'})

## 14. Verify and download the submission file

This imports `assignment02_submission.py` as a separate module, checks that importing it does not train, and calls `load_trained_model(cpu)` + `predict()` exactly as CMS Stage B does. PDF step 3 asks for validation macro recall above 0.80. Unseen patients usually score lower, so higher is safer.

In [ ]:
import importlib.util

submission_path = result['submission']
spec = importlib.util.spec_from_file_location('a02_submission', submission_path)
sub = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sub)  # import only: must not train or download

assert sub.TRAINED_MODEL_B64.strip(), 'TRAINED_MODEL_B64 is empty'
cpu = torch.device('cpu')
sub_model = sub.load_trained_model(cpu)  # exactly what CMS Stage B calls
val_loader = result['val_loader']
y_true = torch.cat([y.long() for _, y in val_loader])
rep = sub.classification_report(y_true, sub.predict(sub_model, val_loader, cpu))

print('Submission file :', submission_path, f'({os.path.getsize(submission_path) / 1e6:.2f} MB)')
print('Conv2d layers   :', sum(isinstance(m, nn.Conv2d) for m in sub_model.modules()))
print(f"Validation      : acc={rep['accuracy']:.4f} macro_recall={rep['macro_recall']:.4f}")
print('Validation macro recall >= 0.80 (PDF step 3):', rep['macro_recall'] >= 0.80)

In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except ImportError:
    print('Not on Colab - the submission file is at:', os.path.abspath(submission_path))